# Schema Validation Demo - Overview

This notebook demonstrates how to validate **schema compliance** between actual data and expected schema definitions using the data quality framework.

## Overview
Schema validation ensures that data conforms to expected structure by verifying:
* Column names match the expected schema
* Data types are correct for each column
* No unexpected columns exist in the data
* All required columns are present

## What This Demo Covers
* Loading test data from CSV files (employees table)
* Defining expected schema with column names and data types
* Checking schema compliance against the expected schema
* Identifying schema violations (missing columns, type mismatches, extra columns)
* Using the `check_schema()` function
* Saving results to a Delta table for tracking and reporting

## Test Scenario
* **Expected Schema**: employees table with columns:
  - employee_id (integer)
  - name (string)
  - department (string)
  - salary (integer)
* **Test Files Available**:
  - employees.csv (has id instead of employee_id, salary is double)
  - employees_missing_cols.csv (missing department and salary columns)
  - employees_wrong_types.csv (employee_id is string, has extra manager_id column)
  - employees_extra_cols.csv (has extra manager_id and department columns)

## Key Functions
* `check_schema(df, expected_schema)` - Validates data schema against expected schema and returns violations

## Output
* Schema validation report showing violations and pass/fail status
* Results saved to `workspace.default.schema_validation_report` table

In [0]:
"""
Schema Validation Check Demo

This script demonstrates how to validate data schema compliance.
It checks that the employees data matches the expected schema definition.

Expected Schema:
    employee_id: integer
    name: string
    department: string
    salary: integer
    
Expected Results:
    - Data may have type mismatches, missing columns, or extra columns
    - Schema violations will be identified and displayed
"""

# ============================================================================
# 1. SETUP: Import required libraries
# ============================================================================
import sys
import os

# ============================================================================
# 2. DYNAMIC PATH CONFIGURATION
# ============================================================================
# Get the base repository path dynamically to avoid hard-coded paths
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# Go up three levels: day35 -> phase2_dq_framework -> notebooks -> data-quality-testing (base)
base_path = os.path.dirname(os.path.dirname(os.path.dirname(notebook_path)))

# ============================================================================
# 3. IMPORT DQ CHECK FUNCTIONS
# ============================================================================
# Add DQ checks module to Python path
checks_path = os.path.join("/Workspace", base_path.lstrip("/"), "src/checks")
sys.path.append(checks_path)
from dq_checks import check_schema

# ============================================================================
# 4. DEFINE EXPECTED SCHEMA
# ============================================================================
# Define the expected schema for the employees table
# Format: list of (column_name, type_name) tuples
# Note: Use PySpark type names (integer, not int)
expected_schema = [
    ("employee_id", "integer"),
    ("name", "string"),
    ("department", "string"),
    ("salary", "integer")
]

print("📋 Expected Schema for Employees Table:")
for col, dtype in expected_schema:
    print(f"  • {col}: {dtype}")

# ============================================================================
# 5. LOAD TEST DATA - EMPLOYEES TABLE
# ============================================================================
# Load employees test data from CSV file
# Available test files for different schema violation scenarios:
#   - employees.csv (has id instead of employee_id, salary is double)
#   - employees_missing_cols.csv (missing department and salary columns)
#   - employees_wrong_types.csv (employee_id is string, has extra manager_id column)
#   - employees_extra_cols.csv (has extra manager_id and department columns)

# Change the filename below to test different scenarios:
test_file = "employees_wrong_types.csv"  # Try different files!

employees_path = os.path.join("/Workspace", base_path.lstrip("/"), f"tests/test_data/{test_file}")
employees = spark.read.option("header", True).option("inferSchema", True).csv(employees_path)

print(f"\n📊 Employees Data (from {test_file}):")
employees.display()

print("\n📋 Actual Schema:")
employees.printSchema()

# ============================================================================
# 6. RUN SCHEMA VALIDATION CHECK
# ============================================================================
# Check schema: Verify that the employees data schema matches the expected schema
# Returns: Dictionary with:
#   - check: "schema_validation"
#   - missing_columns: list of columns in expected but not in actual
#   - extra_columns: list of columns in actual but not in expected
#   - type_mismatches: list of (column, expected_type, actual_type) tuples
#   - passed: boolean indicating if schema matches exactly
result = check_schema(employees, expected_schema)
print("\n🔍 Schema Validation Check Result:")
print(result)

# ============================================================================
# 7. DISPLAY SCHEMA VIOLATIONS IN DETAIL
# ============================================================================
print("\n🚨 Schema Violations:")

if result["missing_columns"]:
    print(f"\n  Missing Columns (in expected schema but not in data):")
    for col in result["missing_columns"]:
        print(f"    • {col}")

if result["extra_columns"]:
    print(f"\n  Extra Columns (in data but not in expected schema):")
    for col in result["extra_columns"]:
        print(f"    • {col}")

if result["type_mismatches"]:
    print(f"\n  Type Mismatches (wrong data type):")
    for col, expected_type, actual_type in result["type_mismatches"]:
        print(f"    • {col}: expected '{expected_type}', got '{actual_type}'")

if result["passed"]:
    print("\n  ✅ No violations found - schema matches perfectly!")

In [0]:
# ============================================================================
# 8. SAVE RESULTS TO DELTA TABLE
# ============================================================================
# Convert the schema validation check result dictionary to a DataFrame
# This allows us to track and analyze schema validation issues over time
result_data = [{
    "check": result["check"],
    "missing_columns": str(result["missing_columns"]),
    "extra_columns": str(result["extra_columns"]),
    "type_mismatches": str(result["type_mismatches"]),
    "passed": result["passed"]
}]

results_df = spark.createDataFrame(result_data)

# Display the results DataFrame before saving
print("\n📋 Schema Validation Check Results Summary:")
results_df.display()

# Save results to Delta table for tracking and reporting
# Table: workspace.default.schema_validation_report
results_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.schema_validation_report")

print("\n✅ Results saved to 'workspace.default.schema_validation_report' table")

In [0]:
# ============================================================================
# 9. VERIFY SAVED RESULTS
# ============================================================================
# Read back the saved schema validation report from Delta table to verify persistence
df = spark.table("workspace.default.schema_validation_report")

print("📊 Reading back from Delta table: workspace.default.schema_validation_report")
df.display()